### loading

In [1]:
import os

In [2]:
# Define a User-Agent string for macOS
os.environ["USER_AGENT"] = "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/114.0.0.0 Safari/537.36"

In [3]:
from langchain_community.document_loaders import WebBaseLoader

In [4]:
#삼엽충의 고도로 복잡한 눈! 
url = "https://creation.kr/question09"

### Crawling

In [5]:
import bs4

In [6]:
loader = WebBaseLoader(
    web_path = url,
    verify_ssl = False,
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("section_wrap  pc_section       side_basic grid_gutter_0 grid_v_gutter_0   ")
        )
    ),
)

In [7]:
page = loader.load()

/opt/miniconda3/envs/llm/lib/python3.9/site-packages/urllib3/connectionpool.py:1099: InsecureRequestWarning: Unverified HTTPS request is being made to host 'creation.kr'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


In [ ]:
page

In [8]:
print(type(page))

<class 'list'>


In [9]:
import re

In [10]:
# Extract URLs that start with "http://creation.kr/"
combined_content = " ".join(doc.page_content for doc in page)
urls = re.findall(r'http://creation\.kr/\S+?view', combined_content)

In [11]:
urls

['http://creation.kr/Circulation/?idx=1295048&bmode=view',
 'http://creation.kr/Dinosaur/?idx=1294603&bmode=view',
 'http://creation.kr/EvidenceofFlood/?idx=1288479&bmode=view',
 'http://creation.kr/Circulation/?idx=1294974&bmode=view',
 'http://creation.kr/Dinosaur/?idx=1294575&bmode=view',
 'http://creation.kr/Circulation/?idx=1295085&bmode=view',
 'http://creation.kr/Circulation/?idx=1793759&bmode=view',
 'http://creation.kr/Dinosaur/?idx=1294502&bmode=view',
 'http://creation.kr/Dinosaur/?idx=1294522&bmode=view',
 'http://creation.kr/Dinosaur/?idx=1294539&bmode=view',
 'http://creation.kr/Burial/?idx=1294400&bmode=view',
 'http://creation.kr/Circulation/?idx=1294949&bmode=view',
 'http://creation.kr/Circulation/?idx=1295081&bmode=view',
 'http://creation.kr/LivingFossils/?idx=1294745&bmode=view',
 'http://creation.kr/Circulation/?idx=1294861&bmode=view',
 'http://creation.kr/Controversy/?idx=1294679&bmode=view',
 'http://creation.kr/Dinosaur/?idx=1294595&bmode=view',
 'http://creat

### Splitting

In [12]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [13]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap=100
)

In [18]:
docs_list = []
for link in urls:
    loader = WebBaseLoader(
        web_path = link,
        verify_ssl = False,
        bs_kwargs=dict(
            parse_only=bs4.SoupStrainer(
                class_=("margin-top-xxl _comment_body_")
            )
        ),
    )
    
    page = loader.load()
    docs = text_splitter.split_documents(page)
    docs_list.append(docs)

/opt/miniconda3/envs/llm/lib/python3.9/site-packages/urllib3/connectionpool.py:1099: InsecureRequestWarning: Unverified HTTPS request is being made to host 'creation.kr'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/opt/miniconda3/envs/llm/lib/python3.9/site-packages/urllib3/connectionpool.py:1099: InsecureRequestWarning: Unverified HTTPS request is being made to host 'creation.kr'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/opt/miniconda3/envs/llm/lib/python3.9/site-packages/urllib3/connectionpool.py:1099: InsecureRequestWarning: Unverified HTTPS request is being made to host 'creation.kr'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/opt/miniconda3/envs/llm/lib/python3.9/s

### Embedding

In [19]:
from langchain_openai import OpenAIEmbeddings

In [20]:
embedding_model = OpenAIEmbeddings(model="text-embedding-3-large")

In [21]:
text_content = [text.page_content for docs in docs_list for text in docs]
embeddings = embedding_model.embed_documents(text_content)

### Vector Store

In [22]:
from langchain.vectorstores import Chroma

In [24]:
# Flatten docs_list if it contains nested lists
docs = [doc for docs in docs_list for doc in docs]

# Use the flattened list with Chroma.from_documents
db = Chroma.from_documents(docs, embedding_model)

### Retrieve

In [27]:
question = "삼엽충 화석은 진화론을 뒷받침하고 있나요?"

In [75]:
retriever = db.as_retriever(
    search_type = "mmr",
    search_kwargs = {"k": 3, "fetch_k": 5, "lambda_mult": 0.8}
)

In [76]:
context = retriever.invoke(question)

In [77]:
context

[Document(metadata={'source': 'http://creation.kr/Textbook/?idx=1289605&bmode=view'}, page_content='중간형태로 주장되는 화석들\xa0: 말, 시조새, 오리너구리, 세이모리아, 익테오스테가임번삼\xa0\xa0 \xa0 \xa0 진화론에서는 무기물에서 유기물이, 유기물에서 단세포로, 단세포에서 다세포로 진화했다고 말한다. 진화의 방향은 단순한 것에서 복잡한 것으로, 저등한 생물이 고등생물로 발전적으로 변화하여 오늘과 같이 다양한 \xa0생물들이 생존하게 되었다고 주장한다. 그리고, 각 단계마다 긴 세월이 소요되었으므로 반드시 무수히 많은 중간종들이 존재하리라고 다윈은 ‘종의 기원’ 개정판(1872)에서 확신있게 피력하였다. 그러나, 각 단계마다 수천만 종의 중간종들이 쏟아져 나와야 함에도 불구하고, 그러한 중간종들은 발견되지 않는다. 더구나, 현재에도 무수한 중간종들이 세계도처에서 태어나야 함에도 불구하고, 단 한건의 사례도 발견되지 않는다. 이것은 엄연한 사실이다. ‘잃어버린 고리는 영원히 잃어버린 것’(Missing links are what we missed)이다. 왜냐하면 처음부터 존재하지 않았기 때문이다.수많은 화석생물을 포함하여 자연계의 모든 생물들은 독자적이고 완벽한 모습으로 나타난다. 중간종의 모습은 그 어느 곳에도 찾아 볼 수 없다. 화석이 발견된 지 일 세기가 지난 현재까지도 중간형태가 보이지 않는다는 것은 화석기록에서 읽을 수 있는 주요한 특징이 되어 있다. [임번삼 외; ibid, p 183]. 모든 생물이 외관상 하등동물에서 고등동물로 진화해 온 듯이 비슷해 보이지만 해부학적인 소견은 이들이 모두 독립적이며 불연속적임을 보여준다. 이같은 사실은 진화론자들에게는 매우 치명적인 약점이다. 그래서, 그들은 생물의 중간종이라고 생각되는 몇 가지 화석들을 진화의 근거로 제시한다. 진화론자들이 중간화석이라고 제시하고 있는 것은 다음과 같이 몇 종에 지나지 않는다. 과연 이들

### Augment

In [78]:
from langchain.prompts import ChatPromptTemplate
from langchain.prompts import HumanMessagePromptTemplate
from langchain_core.messages import SystemMessage

In [79]:
chat_template = ChatPromptTemplate.from_messages(
    [
        SystemMessage(
            content= "당신은 지적설계를 지지하는 과학자입니다."
        ),
        HumanMessagePromptTemplate.from_template(
            """
            {question}
            아래의 문맥에 기반하여 답하여 주세요.
            {context}

            문맥에서 언급되었다는 투의 말을 생략하고 응답을 제공해주세요.
            """
        )
    ]
)

In [80]:
message = chat_template.format_messages(
    question=question,
    context=context[0].page_content
)

### Generate

In [81]:
from langchain_openai import ChatOpenAI

In [82]:
model = ChatOpenAI(
    model_name = "gpt-4o-mini",
    temperature = 0
)

In [83]:
for chunk in model.stream(message):
    print(chunk.content, end="", flush=True)

삼엽충 화석은 진화론을 뒷받침하지 않습니다. 진화론에서는 중간종의 존재가 필수적이라고 주장하지만, 실제로는 그러한 중간종들이 발견되지 않고 있습니다. 화석 기록에서 모든 생물은 독립적이고 완벽한 모습으로 나타나며, 중간형태가 존재하지 않는다는 점은 진화론의 치명적인 약점으로 지적됩니다. 따라서 삼엽충 화석은 진화의 연결고리로서의 역할을 하지 못합니다.

## vector store: Memory가 아닌 disk에 저장시키기

In [86]:
db = Chroma.from_documents(
    documents = docs,
    embedding= embedding_model,
    persist_directory = "./chroma_db"
)